# Autoencoders from scratch

*Part 0 — the primer for the SAE speedrun:
[Part 1 · superposition & SAE basics](01_superposition_and_saes.ipynb) →
[Part 2 · architectures & JumpReLU](02_architectures_jumprelu.ipynb) →
[Part 3 · reconstructing a Neuronpedia dashboard](03_neuronpedia_dashboard.ipynb) →
[Part 4 · training an SAE on Gemma 3 1B](04_train_jumprelu_gemma3_1b.ipynb).*

This primer assumes no autoencoder background at all and builds, in three steps, exactly the
concepts the rest of the series leans on:

1. **The simplest autoencoder, a linear bottleneck** → trained with squared reconstruction
   error, it finds the same answer as PCA (principal component analysis — we derive everything
   from scratch, no PCA background needed).
2. **How does a ReLU enable superposition?** → we justify adding a non-linearity to the
   autoencoder, and show it lets the model exploit sparsity by making errors occur only when
   two features are active at the same time.
3. **How is an SAE different from our dense autoencoders?** → the SAE machine up front
   (encoder, bias, ReLU, decoder — tapping a frozen network's residual stream), then the whole
   round trip with hand-checkable numbers: four named features ("cat", "dog", "French",
   "CAPS") squeezed into two dimensions and un-mixed again.

Everything here runs on CPU in a few seconds — we use tiny toy models so the ground truth is
known and every claim is checkable. The real Gemma 3 work starts in
[Part 3](03_neuronpedia_dashboard.ipynb).


# Two Networks That Look Alike

An autoencoder is one of the oldest ideas in representation learning: squeeze data
through a narrow channel and force it to come out the other side intact. A **sparse
autoencoder** — the tool this series is about — reuses that architecture almost verbatim,
and then inverts the point of it.

Putting them side by side is the fastest way to see what changes. Here is where we are
going — don't worry about decoding every label yet; the whole notebook builds up to this
picture, and we return to it at the end.

<figure style="margin:1.8rem auto;max-width:1000px">
  <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(330px,1fr));
              gap:1.2rem 1.8rem;align-items:start">
    <div>
<svg viewBox="0 0 420 330" role="img" xmlns="http://www.w3.org/2000/svg"
     style="width:100%;height:auto;overflow:visible"
     font-family="system-ui,-apple-system,Segoe UI,sans-serif">
<title>Undercomplete autoencoder: five inputs compressed to a two-unit bottleneck and reconstructed.</title>
<line x1="83" y1="130" x2="194" y2="147" stroke="currentColor" stroke-opacity="0.15"/>
<line x1="83" y1="130" x2="194" y2="189" stroke="currentColor" stroke-opacity="0.15"/>
<line x1="83" y1="168" x2="194" y2="147" stroke="currentColor" stroke-opacity="0.15"/>
<line x1="83" y1="168" x2="194" y2="189" stroke="currentColor" stroke-opacity="0.15"/>
<line x1="83" y1="244" x2="194" y2="147" stroke="currentColor" stroke-opacity="0.15"/>
<line x1="83" y1="244" x2="194" y2="189" stroke="currentColor" stroke-opacity="0.15"/>
<line x1="83" y1="92" x2="194" y2="147" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="92" x2="194" y2="189" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="206" x2="194" y2="147" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="206" x2="194" y2="189" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="226" y1="147" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="147" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="147" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="147" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="147" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="189" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="189" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="189" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="189" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.17"/>
<line x1="226" y1="189" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.17"/>
<text x="210" y="20" text-anchor="middle" font-size="14.5" font-weight="600" fill="currentColor">Undercomplete autoencoder</text>
<text x="210" y="38" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">compresses: 5 → 2 → 5</text>
<text x="140" y="58" text-anchor="middle" font-size="12.5" font-style="italic" fill="currentColor" opacity="0.72">encoder E</text>
<text x="280" y="58" text-anchor="middle" font-size="12.5" font-style="italic" fill="currentColor" opacity="0.72">decoder D</text>
<circle cx="70" cy="92" r="11" fill="#f08c00"/>
<circle cx="70" cy="130" r="11" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="70" cy="168" r="11" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="70" cy="206" r="11" fill="#f08c00"/>
<circle cx="70" cy="244" r="11" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="210" cy="147" r="14" fill="#4c8dd6"/>
<circle cx="210" cy="189" r="14" fill="#4c8dd6"/>
<circle cx="350" cy="92" r="11" fill="#f08c00" fill-opacity="0.42"/>
<circle cx="350" cy="130" r="11" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="350" cy="168" r="11" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="350" cy="206" r="11" fill="#f08c00" fill-opacity="0.42"/>
<circle cx="350" cy="244" r="11" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<text x="70" y="296" text-anchor="middle" font-size="13.5" fill="currentColor">input x</text>
<text x="70" y="313" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">5 numbers</text>
<text x="210" y="296" text-anchor="middle" font-size="13.5" fill="currentColor">bottleneck h</text>
<text x="210" y="313" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">squeezed to 2</text>
<text x="350" y="296" text-anchor="middle" font-size="13.5" fill="currentColor">reconstruction x̂</text>
<text x="350" y="313" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">back to 5</text>
</svg>
    </div>
    <div>
<svg viewBox="0 0 420 330" role="img" xmlns="http://www.w3.org/2000/svg"
     style="width:100%;height:auto;overflow:visible"
     font-family="system-ui,-apple-system,Segoe UI,sans-serif">
<title>Sparse autoencoder: a five-dimensional dense activation expanded into nine latents, only two of which are active, then reconstructed.</title>
<line x1="83" y1="92" x2="200" y2="76" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="92" x2="200" y2="99" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="92" x2="200" y2="145" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="92" x2="200" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="92" x2="200" y2="191" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="92" x2="200" y2="237" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="92" x2="200" y2="260" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="130" x2="200" y2="76" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="130" x2="200" y2="99" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="130" x2="200" y2="145" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="130" x2="200" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="130" x2="200" y2="191" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="130" x2="200" y2="237" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="130" x2="200" y2="260" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="168" x2="200" y2="76" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="168" x2="200" y2="99" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="168" x2="200" y2="145" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="168" x2="200" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="168" x2="200" y2="191" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="168" x2="200" y2="237" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="168" x2="200" y2="260" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="206" x2="200" y2="76" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="206" x2="200" y2="99" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="206" x2="200" y2="145" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="206" x2="200" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="206" x2="200" y2="191" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="206" x2="200" y2="237" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="206" x2="200" y2="260" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="244" x2="200" y2="76" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="244" x2="200" y2="99" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="244" x2="200" y2="145" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="244" x2="200" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="244" x2="200" y2="191" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="244" x2="200" y2="237" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="244" x2="200" y2="260" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="83" y1="92" x2="200" y2="122" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="92" x2="200" y2="214" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="130" x2="200" y2="122" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="130" x2="200" y2="214" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="168" x2="200" y2="122" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="168" x2="200" y2="214" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="206" x2="200" y2="122" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="206" x2="200" y2="214" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="244" x2="200" y2="122" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="83" y1="244" x2="200" y2="214" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="76" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="76" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="76" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="76" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="76" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="99" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="99" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="99" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="99" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="99" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="145" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="145" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="145" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="145" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="145" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="168" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="168" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="168" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="168" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="168" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="191" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="191" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="191" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="191" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="191" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="237" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="237" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="237" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="237" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="237" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="260" x2="337" y2="92" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="260" x2="337" y2="130" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="260" x2="337" y2="168" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="260" x2="337" y2="206" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="260" x2="337" y2="244" stroke="currentColor" stroke-opacity="0.085"/>
<line x1="220" y1="122" x2="337" y2="92" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="122" x2="337" y2="130" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="122" x2="337" y2="168" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="122" x2="337" y2="206" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="122" x2="337" y2="244" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="214" x2="337" y2="92" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="214" x2="337" y2="130" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="214" x2="337" y2="168" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="214" x2="337" y2="206" stroke="#f08c00" stroke-opacity="0.34"/>
<line x1="220" y1="214" x2="337" y2="244" stroke="#f08c00" stroke-opacity="0.34"/>
<text x="210" y="20" text-anchor="middle" font-size="14.5" font-weight="600" fill="currentColor">Sparse (overcomplete) autoencoder</text>
<text x="210" y="38" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">expands: 5 → 9 → 5, but z is sparse</text>
<text x="136" y="58" text-anchor="middle" font-size="12.5" font-style="italic" fill="currentColor" opacity="0.72">encoder W<tspan font-size="9.5" dy="3">enc</tspan><tspan dy="-3">, ReLU</tspan></text>
<text x="286" y="58" text-anchor="middle" font-size="12.5" font-style="italic" fill="currentColor" opacity="0.72">decoder W<tspan font-size="9.5" dy="3">dec</tspan><tspan dy="-3"></tspan></text>
<circle cx="70" cy="92" r="11" fill="#4c8dd6"/>
<circle cx="70" cy="130" r="11" fill="#4c8dd6"/>
<circle cx="70" cy="168" r="11" fill="#4c8dd6"/>
<circle cx="70" cy="206" r="11" fill="#4c8dd6"/>
<circle cx="70" cy="244" r="11" fill="#4c8dd6"/>
<circle cx="210" cy="76" r="8" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="210" cy="99" r="8" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="210" cy="122" r="8" fill="#f08c00"/>
<circle cx="210" cy="145" r="8" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="210" cy="168" r="8" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="210" cy="191" r="8" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="210" cy="214" r="8" fill="#f08c00"/>
<circle cx="210" cy="237" r="8" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="210" cy="260" r="8" fill="none" stroke="currentColor" stroke-opacity="0.5"/>
<circle cx="350" cy="92" r="11" fill="#4c8dd6" fill-opacity="0.42"/>
<circle cx="350" cy="130" r="11" fill="#4c8dd6" fill-opacity="0.42"/>
<circle cx="350" cy="168" r="11" fill="#4c8dd6" fill-opacity="0.42"/>
<circle cx="350" cy="206" r="11" fill="#4c8dd6" fill-opacity="0.42"/>
<circle cx="350" cy="244" r="11" fill="#4c8dd6" fill-opacity="0.42"/>
<text x="70" y="296" text-anchor="middle" font-size="13.5" fill="currentColor">dense activation h</text>
<text x="70" y="313" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">the SAE reads this</text>
<text x="210" y="296" text-anchor="middle" font-size="13.5" fill="currentColor">latents z</text>
<text x="210" y="313" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">9 latents, only 2 active</text>
<text x="350" y="296" text-anchor="middle" font-size="13.5" fill="currentColor">reconstruction ĥ</text>
<text x="350" y="313" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">≈ h</text>
</svg>
    </div>
  </div>
  <div style="display:flex;flex-wrap:wrap;justify-content:center;gap:0.4rem 1.4rem;
              font-size:0.82em;opacity:0.75;margin-top:0.9rem">
    <span style="display:inline-flex;align-items:center;gap:0.4em"><svg width="13" height="13" viewBox="0 0 13 13" aria-hidden="true" style="flex:none"><circle cx="6.5" cy="6.5" r="5.5" fill="#f08c00" fill-opacity="1"/></svg>active sparse feature</span><span style="display:inline-flex;align-items:center;gap:0.4em"><svg width="13" height="13" viewBox="0 0 13 13" aria-hidden="true" style="flex:none"><circle cx="6.5" cy="6.5" r="5.5" fill="#4c8dd6" fill-opacity="1"/></svg>dense / mixed activation</span><span style="display:inline-flex;align-items:center;gap:0.4em"><svg width="13" height="13" viewBox="0 0 13 13" aria-hidden="true" style="flex:none"><circle cx="6.5" cy="6.5" r="5.5" fill="currentColor" fill-opacity="0" stroke="currentColor" stroke-opacity="0.5"/></svg>zero (inactive)</span><span style="display:inline-flex;align-items:center;gap:0.4em"><svg width="26" height="13" viewBox="0 0 26 13" aria-hidden="true" style="flex:none"><circle cx="6.5" cy="6.5" r="5.5" fill="#f08c00" fill-opacity="0.42"/><circle cx="19.5" cy="6.5" r="5.5" fill="#4c8dd6" fill-opacity="0.42"/></svg>reconstructed (approximate)</span>
  </div>
  <figcaption style="font-size:0.9em;opacity:0.75;margin-top:0.9rem;
                     max-width:70ch;margin-left:auto;margin-right:auto;text-align:left">
    <strong>Left:</strong> the classical autoencoder squeezes its input through a narrower
    bottleneck, so it has no choice but to mix features together to survive the squeeze.
    <strong>Right:</strong> the sparse autoencoder runs the same machinery backwards: its
    input is already a dense mixture, its middle layer is <em>wider</em> than its input,
    and what limits it is not a shortage of dimensions but a sparsity constraint — all but
    a handful of latents must be zero. Highlighted edges show the subnetwork
    actually carrying signal.
  </figcaption>
</figure>


# The simplest autoencoder: an undercomplete linear autoencoder

The simplest autoencoder is an **undercomplete linear autoencoder**: a linear encoder compresses the input into a lower-dimensional latent representation, and a linear decoder reconstructs the input from that representation. Specifically, we define:
- **linear**: both encoder and decoder are linear maps;
- **undercomplete**: the latent dimension is smaller than the input dimension, $m<n$;
- **autoencoder**: it is trained to reconstruct its input.

Let the input be

$$
x \in \mathbb{R}^n
$$

and let its lower-dimensional representation be

$$
h \in \mathbb{R}^m,
\qquad m<n.
$$

The encoder maps the input into the bottleneck:

$$
h = Ex,
\qquad
E \in \mathbb{R}^{m \times n}.
$$

The decoder maps the bottleneck back into the original space:

$$
\hat{x} = Dh,
\qquad
D \in \mathbb{R}^{n \times m}.
$$

Combining the encoder and decoder gives

$$
\hat{x} = DEx = Ax,
$$

where

$$
A = DE \in \mathbb{R}^{n \times n}.
$$

The full computation is therefore

$$
\mathbb{R}^n
\xrightarrow{E}
\mathbb{R}^m
\xrightarrow{D}
\mathbb{R}^n.
$$

Although $A$ maps an $n$-dimensional input to an $n$-dimensional output, the information must pass through an $m$-dimensional bottleneck. This means

$$
\operatorname{rank}(A) \leq m.
$$

Another way to see this is to consider the decoder. Every reconstruction has the form

$$
\hat{x} = Dh.
$$

Therefore, every reconstruction is a linear combination of the $m$ columns of $D$. Every possible output must lie in the column space of $D$:

$$
\hat{x} \in \operatorname{col}(D).
$$

Since $D$ has $m$ columns,

$$
\dim\left(\operatorname{col}(D)\right) \leq m.
$$

Thus, the linear autoencoder must reconstruct every input using a single shared subspace of dimension at most $m$.

Training the autoencoder means choosing $E$ and $D$, or equivalently choosing a linear map $A=DE$ with rank at most $m$, to minimise the average squared reconstruction error:

$$
\min_{\operatorname{rank}(A)\leq m}
\mathbb{E}_{x}
\left[
\lVert x-Ax\rVert_2^2
\right].
$$

In other words, the linear autoencoder tries to find the best $m$-dimensional linear subspace for reconstructing the data.


## Linear autoencoders recover the principal components

Suppose the datapoints are **centred** — each feature's mean has been subtracted, so the data
cloud sits at the origin (this is the setting PCA is defined in) — and stored as the columns of

$$
X \in \mathbb{R}^{n \times N},
$$

where $n$ is the number of features and $N$ is the number of datapoints.

An undercomplete linear autoencoder reconstructs the data as

$$
\hat{X} = DEX = AX,
$$

where

$$
E \in \mathbb{R}^{m \times n},
\qquad
D \in \mathbb{R}^{n \times m},
\qquad
A = DE,
$$

and $m<n$. Since $A$ factors through an $m$-dimensional bottleneck,

$$
\operatorname{rank}(A) \leq m.
$$

Training the autoencoder with squared reconstruction error is equivalent to solving

$$
\min_{\operatorname{rank}(A)\leq m}
\lVert X-AX\rVert_F^2,
$$

where the Frobenius norm $\lVert\cdot\rVert_F$ is just the elementwise $L_2$ norm: square every
entry, sum, take the root.

### The truncated SVD

Three standard facts, stated without proof:

1. Any matrix factors as $X = U\Sigma V^\top$ — the **singular value decomposition (SVD)** —
   with orthonormal columns in $U$ and $V$, and $\Sigma$ diagonal with non-negative entries
   $\sigma_1 \geq \sigma_2 \geq \dots$, the **singular values**.
2. The **Eckart–Young theorem**: a best rank-$m$ approximation to $X$ (in Frobenius norm) is
   obtained by keeping the top $m$ singular directions and discarding the rest,
   $$X_m = U_m\Sigma_m V_m^\top,$$
   where $U_m$ contains the first $m$ left singular vectors. (It is *the* best when
   $\sigma_m > \sigma_{m+1}$.)
3. For centred data, the columns of $U$ are exactly the **principal components** — the
   orthogonal directions of greatest variance in the data, in decreasing order.

Now define

$$
A^* = U_mU_m^\top.
$$

This is the orthogonal projection onto the subspace spanned by the first $m$ left singular
vectors. Applying it to the data gives

$$
A^*X = U_mU_m^\top U\Sigma V^\top.
$$

Because the columns of $U$ are orthonormal, $U_m^\top U$ selects the first $m$ rows of
$\Sigma V^\top$. Therefore,

$$
A^*X = U_m\Sigma_mV_m^\top = X_m,
\qquad\text{i.e.}\qquad
\boxed{A^*X=X_m}.
$$

And no rank-$\leq m$ map can do better: $\operatorname{rank}(A) \leq m$ forces
$\operatorname{rank}(AX) \leq m$, so Eckart–Young says
$\lVert X-AX\rVert_F \geq \lVert X-X_m\rVert_F$ — and $A^*$ attains that bound.

The optimal linear autoencoder therefore produces exactly the truncated-SVD approximation of
the centred data. Its bottleneck preserves the subspace spanned by the first $m$ principal
components.

### Encoder and decoder weights

One optimal solution is

$$
E = U_m^\top,
\qquad
D = U_m.
$$

The latent representation $h = U_m^\top x$ contains the coordinates of $x$ along the first $m$
principal directions, and the reconstruction is $\hat{x} = U_mU_m^\top x$. This solution has
**tied weights**, $D = E^\top$ — the decoder is the encoder's transpose, so a single matrix
serves both directions. (We reuse this trick in the next section.)

The split into $E$ and $D$ is not unique, though: for any invertible
$R\in\mathbb{R}^{m\times m}$, the pair $E = RU_m^\top$, $D = U_mR^{-1}$ has the same product
$DE = U_mU_m^\top$. The coordinate system *inside* the bottleneck is a free choice that
reconstruction alone cannot pin down. Hold that thought — a sparse autoencoder faces exactly
the same freedom, and it is the *sparsity* of its latents that breaks the tie and picks out
one privileged basis.

### Minimal numerical example

Rank-2 data through a rank-1 bottleneck — let's verify the boxed claim numerically.


In [ ]:
import numpy as np

# Centred datapoints stored as columns
X = np.array([
    [-2.0, -1.0,  1.0,  2.0],
    [-1.8, -0.9,  0.9,  1.8],
    [ 0.2, -0.1,  0.1, -0.2],
]) # rank 2 matrix

m = 1

# Singular value decomposition
U, singular_values, Vt = np.linalg.svd(X, full_matrices=False)

# Keep the first m singular directions
U_m = U[:, :m]
S_m = np.diag(singular_values[:m])
Vt_m = Vt[:m, :]

# Truncated-SVD approximation
X_m = U_m @ S_m @ Vt_m

# Linear-autoencoder reconstruction
E = U_m.T
D = U_m
X_hat = D @ E @ X

# This autoencoder has a one-dimensional bottleneck and constructs the best rank-1 approximation $X_1$ to the rank-2 matrix $X$.
print(np.allclose(X_hat, X_m))


True


This prints

```text
True
```

because both procedures compute

$$
X_m = U_mU_m^\top X.
$$

The main result is therefore:

$$
\boxed{
\text{undercomplete linear autoencoder}
\quad\Longleftrightarrow\quad
\text{projection onto the principal-component subspace}
}
$$

(for centred data; equivalently, give the encoder and decoder bias terms).

This equivalence is also the linear autoencoder's ceiling: with
$\operatorname{rank}(A)\leq m$ it can never represent more features than it has bottleneck
dimensions — anything outside the top-$m$ subspace is simply lost. But real features are
**sparse**: most of them are off most of the time. A single nonlinearity lets a bottleneck
cheat that ceiling and store *more* features than dimensions. That trick is called
**superposition**, and the next section builds it in the smallest example possible.


# How do ReLUs enable superposition?

**Superposition** means storing more features than you have dimensions, by letting the
features share directions in activation space. We just proved a *linear* bottleneck cannot do
this — $\operatorname{rank}(A) \leq m$ — so something nonlinear has to give. Here is the
smallest possible example.

## The setup

A tiny tied-weight autoencoder: two input features squeezed through a **one-dimensional**
bottleneck and back out again.

$$
x_A,\, x_B \;\longrightarrow\; h \;\longrightarrow\; \hat{x}_A,\, \hat{x}_B
$$

$$
h = Wx, \qquad \hat{x} = W^\top h = W^\top W x
$$

## Sparsity

Consider that our data is **sparse**: features $x_A$ and $x_B$ are rarely on at the same time
(they rarely "co-fire"). So in practice

$$
x \approx \begin{bmatrix} 1 \\ 0 \end{bmatrix}
\quad \text{or} \quad
\begin{bmatrix} 0 \\ 1 \end{bmatrix}
$$

In $x$-space the data lives (almost) entirely on the two axes.

## The bottleneck

Take $W = \begin{bmatrix} 1 & -1 \end{bmatrix}$:

$$
h = Wx = \begin{bmatrix} 1 & -1 \end{bmatrix}
\begin{bmatrix} x_A \\ x_B \end{bmatrix}
= x_A - x_B
$$

$$
x = \begin{bmatrix} 1 \\ 0 \end{bmatrix} \Rightarrow h = 1,
\qquad
x = \begin{bmatrix} 0 \\ 1 \end{bmatrix} \Rightarrow h = -1
$$

**The sign of $h$ tells us which feature is active.** The two features land at $+1$ and $-1$,
on opposite sides of the origin, and one scalar is carrying two features — that's the
superposition.

## The problem: interference

Because our weights are tied, $\hat{x} = W^\top W x$:

$$
\begin{bmatrix} \hat{x}_A \\ \hat{x}_B \end{bmatrix}
=
\begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}
\begin{bmatrix} x_A \\ x_B \end{bmatrix}
=
\begin{bmatrix} x_A - x_B \\ x_B - x_A \end{bmatrix}
$$

But this causes problems:

$$
x = \begin{bmatrix} 1 \\ 0 \end{bmatrix}
\;\Rightarrow\; h = 1
\;\Rightarrow\; \hat{x} = \begin{bmatrix} 1 \\ -1 \end{bmatrix}
\qquad \text{✗ wrong!}
$$

The $-1$ is an **interference** term: the decoder cannot help but read the "$A$ is on" signal
as "$B$ is negatively on". (The culprit is the rank-1 bottleneck itself, not the weight tying —
an untied linear decoder fails in exactly the same way.)

## The fix

If we add a ReLU:

$$
\mathrm{ReLU}\!\left(\begin{bmatrix} 1 \\ -1 \end{bmatrix}\right)
= \begin{bmatrix} 1 \\ 0 \end{bmatrix}
\qquad \text{✓ correct!}
$$

The interference lands on the *negative* side — not by luck, but by construction: we embedded
the two features at $+1$ and $-1$, pointing in opposite directions, precisely so that each
feature's bleed-through into the other comes out negative. (Trained toy models discover the
same antipodal arrangement on their own — [Part 1](01_superposition_and_saes.ipynb) shows it happening.) Features themselves are
non-negative, so the nonlinearity clips the interference away:

$$
\hat{x} = \mathrm{ReLU}\left(W^\top W x\right)
$$

One detail makes the answer come out *exactly* right: the columns of $W$ have unit norm, so
the surviving coordinate is already at the correct scale. ReLU deletes negative interference —
it does not rescale what remains.

## Where it still breaks

| input | $x$ | $h = x_A - x_B$ | $\hat{x} = W^\top W x$ | $\hat{x} = \mathrm{ReLU}(W^\top W x)$ |
|---|---|---|---|---|
| none | $(0,0)$ | $0$ | $(0,0)$ ✓ | $(0,0)$ ✓ |
| $A$ only | $(1,0)$ | $+1$ | $(1,-1)$ ✗ | $(1,0)$ ✓ |
| $B$ only | $(0,1)$ | $-1$ | $(-1,1)$ ✗ | $(0,1)$ ✓ |
| both | $(1,1)$ | $0$ | $(0,0)$ ✗ | $(0,0)$ ✗ |

**ReLU is still wrong when features co-fire, but is correct otherwise.** The input $(1,1)$ is
indistinguishable from $(0,0)$ the moment it enters the bottleneck — $h = 0$ for both — and no
decoder can recover information that was destroyed upstream.

## Takeaway

When your features are sparse (they rarely co-fire), you can use a bottleneck to compress
them — but the compression creates interference. A ReLU removes the harmful *negative*
interference; what remains are the collisions, which only occur when features fire together.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# h    = W x            with W = [1, -1]
# xhat = W^T h          (linear)  or  ReLU(W^T h)
W = np.array([[1.0, -1.0]])          # shape (1, 2)

labels = ["none", "A only", "B only", "both"]
X = np.array([
    [0.0, 0.0],   # none:   nothing on
    [1.0, 0.0],   # A only: just feature x_A
    [0.0, 1.0],   # B only: just feature x_B
    [1.0, 1.0],   # both:   features co-fire, violates sparsity
])
colors = ["#444444", "#d1495b", "#2e6f9e", "#7b9e2e"]

H         = X @ W.T                  # (4, 1)  bottleneck code
Xhat_lin  = H @ W                    # (4, 2)  xhat = W^T W x
Xhat_relu = np.maximum(Xhat_lin, 0)  # (4, 2)  xhat = ReLU(W^T W x)

for lab, x, h, xl, xr in zip(labels, X, H, Xhat_lin, Xhat_relu):
    print(f"{lab:7s}: x={x}  h={h[0]:+.0f}  xhat_lin={xl}  xhat_relu={xr}")


def scatter2d(ax, pts, title, xlabel, ylabel):
    ax.axhline(0, color="0.75", lw=0.8, zorder=0)
    ax.axvline(0, color="0.75", lw=0.8, zorder=0)
    seen = {}
    for lab, (px, py), c in zip(labels, pts, colors):
        ax.scatter(px, py, s=90, color=c, zorder=3)
        k = (round(px, 6), round(py, 6))          # stack labels when points coincide
        seen[k] = seen.get(k, 0) + 1
        ax.annotate(lab, (px, py), textcoords="offset points",
                    xytext=(8, 8 + 14 * (seen[k] - 1)), color=c,
                    fontsize=11, fontweight="bold")
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6)
    ax.set_aspect("equal")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.25, ls=":")


def scatter1d(ax, vals, title, xlabel):
    ax.axhline(0, color="0.4", lw=1.2, zorder=1)
    seen = {}
    for lab, v, c in zip(labels, vals.ravel(), colors):
        ax.scatter(v, 0, s=90, color=c, zorder=3)
        k = round(v, 6)
        seen[k] = seen.get(k, 0) + 1
        ax.annotate(lab, (v, 0), textcoords="offset points",
                    xytext=(8, 8 + 14 * (seen[k] - 1)), color=c,
                    fontsize=11, fontweight="bold")
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.0, 1.0)
    ax.set_yticks([]); ax.set_xticks([-1, 0, 1])
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=11)
    ax.spines[["left", "right", "top"]].set_visible(False)


fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5))

scatter2d(axes[0, 0], X, "1. $x$-space (input)", "$x_A$", "$x_B$")
scatter1d(axes[0, 1], H, "2. $h$-space (bottleneck), $h = x_A - x_B$", "$h$")
scatter2d(axes[1, 0], Xhat_lin,
          r"3. $\hat{x} = W^\top W x$  (no ReLU)", r"$\hat{x}_A$", r"$\hat{x}_B$")
scatter2d(axes[1, 1], Xhat_relu,
          r"4. $\hat{x} = \mathrm{ReLU}(W^\top W x)$", r"$\hat{x}_A$", r"$\hat{x}_B$")

fig.suptitle("How ReLUs enable superposition (2 features, 1 hidden unit, tied weights)",
             fontsize=13, y=0.98)
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

Reading the panels left-to-right, top-to-bottom: the four inputs start out distinct
in $x$-space; the bottleneck collapses **none** and **both** onto the same value; the linear
decoder pushes **A only** and **B only** off-axis into negative-interference territory; and
the ReLU clips them back onto the correct axes, leaving only the none/both collision — the one
error that was baked in before the decoder ever ran.

Everything so far ran in the *compression* direction: we knew the sparse features $x$ and
watched them get squeezed into a dense $h$. A real network is the same story at scale — its
activations are the dense $h$ — except that nobody hands us the sparse features that produced
them. The last step of this primer is to run the story backwards: given only $h$, infer which
features were active.


# How is an SAE different? Running the map backwards

Note the reuse about to happen: the compressed code $h$ of our autoencoders becomes the
**input** of the sparse autoencoder. Interpretability hands us the middle of someone else's
autoencoder — the dense activations of a trained network — and asks what is stored in there.
Let's run the whole round trip with numbers small enough to check by hand.

## The machine, up front

In case you skipped straight to this section: a sparse autoencoder is a **one-hidden-layer
autoencoder bolted onto a trained, frozen network**. Pick a layer; at every token, tap the
residual-stream activation $h \in \mathbb{R}^m$ and feed it through an encoder and a decoder:

$$
z = \operatorname{ReLU}(W_{\text{enc}} h + b_{\text{enc}}),
\qquad
\hat h = W_{\text{dec}}\, z + b_{\text{dec}},
$$

with $W_{\text{enc}} \in \mathbb{R}^{M \times m}$ and $W_{\text{dec}} \in \mathbb{R}^{m \times M}$
— and, unusually for an autoencoder, $M > m$: the hidden layer is *wider* than the input.
Training minimises reconstruction error plus a sparsity penalty,

$$
\mathcal{L} = \lVert h - \hat h \rVert_2^2 + \lambda \lVert z \rVert_1.
$$

<figure style="margin:1.8rem auto;text-align:center;max-width:900px">
<svg viewBox="0 0 860 480" role="img" xmlns="http://www.w3.org/2000/svg" style="width:100%;height:auto;overflow:visible" font-family="system-ui,-apple-system,Segoe UI,sans-serif">
<title>A sparse autoencoder bolted on top of a frozen network: the residual stream runs along the bottom; the tapped activation h (3 dimensions) is expanded by the encoder into a much taller latent vector z (11 dimensions, only 2 active), then contracted by the decoder back to a reconstruction of h.</title>
<defs><marker id="arrM" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="currentColor" fill-opacity="0.55"/></marker></defs>
<text x="430" y="24" text-anchor="middle" font-size="14.5" font-weight="600" fill="currentColor">The machine: an SAE tapping a residual stream</text>
<line x1="15" y1="420" x2="30" y2="420" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.6"/>
<line x1="125" y1="420" x2="150" y2="420" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.6"/>
<line x1="245" y1="420" x2="330" y2="420" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.6"/>
<line x1="425" y1="420" x2="845" y2="420" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.6" marker-end="url(#arrM)"/>
<rect x="30" y="398" width="95" height="44" rx="8" fill="currentColor" fill-opacity="0.06" stroke="currentColor" stroke-opacity="0.4"/>
<text x="77.5" y="425" text-anchor="middle" font-size="12" fill="currentColor" opacity="0.75">block ℓ−1</text>
<rect x="150" y="398" width="95" height="44" rx="8" fill="currentColor" fill-opacity="0.06" stroke="currentColor" stroke-opacity="0.4"/>
<text x="197.5" y="425" text-anchor="middle" font-size="12" fill="currentColor" opacity="0.75">block ℓ</text>
<rect x="330" y="398" width="95" height="44" rx="8" fill="currentColor" fill-opacity="0.06" stroke="currentColor" stroke-opacity="0.4"/>
<text x="377.5" y="425" text-anchor="middle" font-size="12" fill="currentColor" opacity="0.75">block ℓ+1</text>
<text x="645" y="408" text-anchor="middle" font-size="11" font-style="italic" fill="currentColor" opacity="0.6">residual stream (frozen network) →</text>
<circle cx="258" cy="420" r="7" fill="#4c8dd6"/>
<text x="258" y="464" text-anchor="middle" font-size="11" fill="currentColor" opacity="0.7">tap h after block ℓ</text>
<line x1="258" y1="411" x2="258" y2="240" stroke="currentColor" stroke-opacity="0.6" stroke-width="1.8" marker-end="url(#arrM)"/>
<rect x="245" y="170" width="26" height="18" rx="3" fill="#4c8dd6"/>
<rect x="245" y="192" width="26" height="18" rx="3" fill="#4c8dd6"/>
<rect x="245" y="214" width="26" height="18" rx="3" fill="#4c8dd6"/>
<rect x="417" y="90" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="110" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="130" width="26" height="16" rx="3" fill="#f08c00"/>
<rect x="417" y="150" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="170" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="190" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="210" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="230" width="26" height="16" rx="3" fill="#f08c00"/>
<rect x="417" y="250" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="270" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="417" y="290" width="26" height="16" rx="3" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<rect x="589" y="170" width="26" height="18" rx="3" fill="#4c8dd6" fill-opacity="0.42"/>
<rect x="589" y="192" width="26" height="18" rx="3" fill="#4c8dd6" fill-opacity="0.42"/>
<rect x="589" y="214" width="26" height="18" rx="3" fill="#4c8dd6" fill-opacity="0.42"/>
<polygon points="271,170 271,232 417,306 417,90" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.22"/>
<polygon points="443,90 443,306 589,232 589,170" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.22"/>
<text x="344" y="182" text-anchor="middle" font-size="11.5" fill="currentColor">ReLU(W<tspan font-size="9" dy="3">enc</tspan><tspan dy="-3"> h + </tspan>b<tspan font-size="9" dy="3">enc</tspan><tspan dy="-3">)</tspan></text>
<text x="344" y="200" text-anchor="middle" font-size="10.5" font-style="italic" fill="currentColor" opacity="0.6">encoder — expand</text>
<text x="516" y="182" text-anchor="middle" font-size="11.5" fill="currentColor">W<tspan font-size="9" dy="3">dec</tspan><tspan dy="-3"> z + </tspan>b<tspan font-size="9" dy="3">dec</tspan><tspan dy="-3"></tspan></text>
<text x="516" y="200" text-anchor="middle" font-size="10.5" font-style="italic" fill="currentColor" opacity="0.6">decoder — contract</text>
<text x="258" y="156" text-anchor="middle" font-size="12.5" fill="currentColor">h ∈ ℝᵐ</text>
<text x="430" y="52" text-anchor="middle" font-size="12.5" fill="currentColor">z ∈ ℝᴹ</text>
<text x="430" y="70" text-anchor="middle" font-size="11" fill="currentColor" opacity="0.65">M ≫ m, mostly zeros — bar height = dimensions</text>
<text x="602" y="156" text-anchor="middle" font-size="12.5" fill="currentColor">ĥ ∈ ℝᵐ</text>
<line x1="602" y1="238" x2="268" y2="414" stroke="currentColor" stroke-opacity="0.5" stroke-dasharray="5 4" marker-end="url(#arrM)"/>
<text x="452" y="342" text-anchor="middle" font-size="11.5" fill="currentColor">trained so ĥ ≈ h</text>
<text x="452" y="360" text-anchor="middle" font-size="11" fill="currentColor" opacity="0.8">L = ‖h − ĥ‖₂² + λ‖z‖₁</text>
</svg>
<figcaption style="font-size:0.9em;opacity:0.72;margin-top:0.5rem;
                   max-width:70ch;margin-left:auto;margin-right:auto;text-align:left">
The network being studied runs along the bottom and stays <em>frozen</em> — the SAE is a
probe bolted on top at one chosen layer, changing nothing about the model. Bar height =
number of dimensions: the encoder (weights, bias, ReLU) <em>expands</em> the tapped
activation <em>h</em> into a latent vector <em>z</em> with far more dimensions, almost
all of them zero; the decoder contracts it back to a reconstruction ĥ. Training pushes
ĥ toward <em>h</em> while the λ‖z‖₁ term keeps <em>z</em> sparse.
</figcaption>
</figure>

Two choices should look strange after the last two sections: the middle layer is *bigger* than
the input (what kind of bottleneck is that?), and sparsity is imposed by an extra loss term
instead of being a property of the data we feed in. The rest of this section justifies both by
running one concrete example end to end.

## Step 1 — the network compresses (superposition)

Suppose the network reads text, and there are $n = 4$ semantic features it wants to track:

| | feature | fires on |
| --- | --- | --- |
| 1 | **cat** | text about cats |
| 2 | **dog** | text about dogs |
| 3 | **French** | text written in French |
| 4 | **CAPS** | TEXT WRITTEN IN ALL CAPS |

These are exactly the kind of features real language models track: each is **sparse** (most
text is not about cats) and they **co-occur freely** (a French sentence about a cat is
perfectly ordinary). But suppose the network has only $m = 2$ residual-stream dimensions to
store them in. Each feature gets a *direction* in that 2-D space; stack the four directions as
the columns of

$$
W_{\text{true}} =
\begin{pmatrix}
1 & 0 & 0.7 & 0.7\\
0 & 1 & 0.7 & -0.7
\end{pmatrix},
\qquad
d_1 = \begin{pmatrix}1\\0\end{pmatrix},\;
d_2 = \begin{pmatrix}0\\1\end{pmatrix},\;
d_3 = \begin{pmatrix}0.7\\0.7\end{pmatrix},\;
d_4 = \begin{pmatrix}0.7\\-0.7\end{pmatrix},
$$

so $d_1$ is the **cat** direction, $d_2$ the **dog** direction, $d_3$ the **French**
direction, and $d_4$ the **CAPS** direction. Four directions cannot all be orthogonal in two
dimensions — they must overlap. That is superposition again, just $4 \to 2$ instead of
$2 \to 1$.

From here on we write $s$ for the vector of true feature activations — the $x$ of the previous
section — to emphasise that, from where we now stand, it is *hidden*. Now run the network over
the sentence *« le chat »*. At the token **chat**, the **cat** and **French** features are on;
**dog** and **CAPS** are off:

$$
s = \begin{pmatrix}1\\0\\1\\0\end{pmatrix}
\begin{matrix}\leftarrow \text{cat}\\ \leftarrow \text{dog}\\ \leftarrow \text{French}\\ \leftarrow \text{CAPS}\end{matrix}
$$

The residual-stream activation the network actually carries at that token is the sum of the
active directions:

$$
h = W_{\text{true}}\, s = d_{\text{cat}} + d_{\text{French}} =
\begin{pmatrix}1\\0\end{pmatrix} + \begin{pmatrix}0.7\\0.7\end{pmatrix} =
\begin{pmatrix}1.7\\0.7\end{pmatrix}.
$$

$$
\underbrace{\begin{pmatrix}1\\0\\1\\0\end{pmatrix}}_{\text{sparse features } s}
\;\longrightarrow\;
\underbrace{\begin{pmatrix}1.7\\0.7\end{pmatrix}}_{\text{dense activation } h}
$$

## Coordinates are not features

Look at $h = (1.7,\, 0.7)$. **Both coordinates are nonzero** — but that does *not* mean two
features are active, and the coordinates certainly don't tell you *which* ones. There is no
"cat coordinate" and no "French coordinate": the first coordinate reads $1.7$ because **cat**
contributed $1$ and **French** contributed $0.7$, and a single feature like **French**
($d_3 = (0.7,\, 0.7)$) lights up *both* coordinates on its own.

The coordinates of $h$ are just the model's internal storage basis. The sparsity claim is not
"the coordinates of $h$ are usually zero" — they usually aren't. The claim is:

> there exists a *different* set of directions ($d_{\text{cat}}, \dots, d_{\text{CAPS}}$)
> whose activation coefficients ($s$) are sparse.

The SAE's job is to find those directions.

## Step 2 — the SAE decompresses (sparse inference)

The SAE is handed only $h = (1.7,\, 0.7)$. It never sees $s$ or $W_{\text{true}}$ — it doesn't
even know the network is tracking cats. It computes the encoder–decoder pair from the top
of the section:

$$
z = \operatorname{ReLU}(W_{\text{enc}} h + b_{\text{enc}}),
\qquad
\hat h = W_{\text{dec}}\, z + b_{\text{dec}},
$$

and it is trained so that $\hat h \approx h$ while $z$ stays sparse. If training succeeds, the
columns of $W_{\text{dec}}$ converge to the true feature directions —
$W_{\text{dec}} \approx W_{\text{true}}$ — and on our example

$$
z \approx \begin{pmatrix}1\\0\\1\\0\end{pmatrix} = s,
\qquad
\hat h = W_{\text{dec}}\, z \approx d_{\text{cat}} + d_{\text{French}} = h.
$$

The SAE has *un-mixed* the activation: two numbers in, four numbers out, only two of them
nonzero. Latent 1 has become a **cat detector** and latent 3 a **French detector** — each
nonzero latent names one feature, which is exactly the kind of labelled, monosemantic latent
you will browse on the Neuronpedia dashboard in [Part 3](03_neuronpedia_dashboard.ipynb).


<figure style="margin:1.8rem auto;max-width:1000px">
  <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(330px,1fr));
              gap:1.2rem 1.8rem;align-items:center">
    <div>
<svg viewBox="0 0 420 340" role="img" xmlns="http://www.w3.org/2000/svg"
     style="width:100%;height:auto;overflow:visible"
     font-family="system-ui,-apple-system,Segoe UI,sans-serif">
<title>Four feature directions in a two-dimensional residual stream; the activation h is the sum of the two active directions.</title>
<defs>
<marker id="arrO" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#f08c00"/></marker>
<marker id="arrB" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#4c8dd6"/></marker>
<marker id="arrG" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="currentColor" fill-opacity="0.45"/></marker>
</defs>
<text x="210" y="22" text-anchor="middle" font-size="14.5" font-weight="600" fill="currentColor">Four feature directions, two dimensions</text>
<!-- axes -->
<line x1="24" y1="210" x2="400" y2="210" stroke="currentColor" stroke-opacity="0.18"/>
<line x1="110" y1="322" x2="110" y2="46" stroke="currentColor" stroke-opacity="0.18"/>
<!-- inactive directions d2, d4 -->
<line x1="110" y1="210" x2="110" y2="129" stroke="currentColor" stroke-opacity="0.45" marker-end="url(#arrG)"/>
<line x1="110" y1="210" x2="167" y2="266" stroke="currentColor" stroke-opacity="0.45" marker-end="url(#arrG)"/>
<text x="80" y="120" text-anchor="middle" font-size="12.5" fill="currentColor" opacity="0.6">d₂ “dog”</text>
<text x="184" y="282" text-anchor="middle" font-size="12.5" fill="currentColor" opacity="0.6">d₄ “CAPS”</text>
<!-- active directions d1, d3 -->
<line x1="110" y1="210" x2="191" y2="210" stroke="#f08c00" stroke-width="2.4" marker-end="url(#arrO)"/>
<line x1="110" y1="210" x2="166" y2="155" stroke="#f08c00" stroke-width="2.4" marker-end="url(#arrO)"/>
<text x="163" y="230" text-anchor="middle" font-size="12.5" font-weight="600" fill="#f08c00">d₁ “cat”</text>
<text x="152" y="141" text-anchor="middle" font-size="12.5" font-weight="600" fill="#f08c00">d₃ “French”</text>
<!-- tip-to-tail sum: d3 translated to tip of d1 -->
<line x1="195" y1="210" x2="251" y2="156" stroke="#f08c00" stroke-width="1.6" stroke-dasharray="5 4" stroke-opacity="0.75" marker-end="url(#arrO)"/>
<text x="238" y="196" text-anchor="middle" font-size="11.5" fill="#f08c00" opacity="0.85">+ d₃</text>
<!-- h -->
<line x1="110" y1="210" x2="250" y2="152" stroke="#4c8dd6" stroke-width="3.2" marker-end="url(#arrB)"/>
<circle cx="255" cy="150" r="4.5" fill="#4c8dd6"/>
<text x="272" y="134" text-anchor="start" font-size="13" font-weight="600" fill="#4c8dd6">h = d₁ + d₃</text>
<text x="272" y="151" text-anchor="start" font-size="12" fill="#4c8dd6" opacity="0.85">= “cat” + “French”</text>
<text x="272" y="168" text-anchor="start" font-size="12" fill="#4c8dd6" opacity="0.85">= (1.7, 0.7)</text>
<!-- axis labels -->
<text x="392" y="228" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.55">h¹</text>
<text x="94" y="54" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.55">h²</text>
<text x="210" y="336" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">both coordinates of h are nonzero — yet only “cat” and “French” are actually on</text>
</svg>
    </div>
    <div>
<svg viewBox="0 0 540 340" role="img" xmlns="http://www.w3.org/2000/svg"
     style="width:100%;height:auto;overflow:visible"
     font-family="system-ui,-apple-system,Segoe UI,sans-serif">
<title>Round trip with numbers: sparse features (1,0,1,0) become the dense activation (1.7,0.7); the SAE infers sparse latents (1,0,1,0) and reconstructs the activation.</title>
<defs>
<marker id="arrF" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="currentColor" fill-opacity="0.55"/></marker>
</defs>
<text x="270" y="22" text-anchor="middle" font-size="14.5" font-weight="600" fill="currentColor">The round trip, with numbers</text>
<!-- s column (4 cells) -->
<g font-size="13" font-weight="600" text-anchor="middle">
<rect x="42" y="100" width="44" height="34" rx="6" fill="#f08c00"/><text x="64" y="122" fill="#fff">1</text>
<rect x="42" y="138" width="44" height="34" rx="6" fill="none" stroke="currentColor" stroke-opacity="0.4"/><text x="64" y="160" fill="currentColor" opacity="0.5" font-weight="400">0</text>
<rect x="42" y="176" width="44" height="34" rx="6" fill="#f08c00"/><text x="64" y="198" fill="#fff">1</text>
<rect x="42" y="214" width="44" height="34" rx="6" fill="none" stroke="currentColor" stroke-opacity="0.4"/><text x="64" y="236" fill="currentColor" opacity="0.5" font-weight="400">0</text>
</g>
<text x="64" y="282" text-anchor="middle" font-size="13" fill="currentColor">s</text>
<text x="64" y="299" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.62">sparse features</text>
<text x="64" y="312" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.62">(hidden from us)</text>
<g font-size="9.5" text-anchor="end" fill="currentColor" opacity="0.65">
<text x="36" y="121">cat</text><text x="36" y="159">dog</text><text x="36" y="197">French</text><text x="36" y="235">CAPS</text>
</g>
<!-- arrow 1 -->
<line x1="94" y1="174" x2="158" y2="174" stroke="currentColor" stroke-opacity="0.55" stroke-width="1.6" marker-end="url(#arrF)"/>
<text x="127" y="160" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.7">the network</text>
<text x="127" y="192" text-anchor="middle" font-size="10.5" font-style="italic" fill="currentColor" opacity="0.7">superposition</text>
<!-- h column (2 cells) -->
<g font-size="13" font-weight="600" text-anchor="middle">
<rect x="168" y="138" width="52" height="34" rx="6" fill="#4c8dd6"/><text x="194" y="160" fill="#fff">1.7</text>
<rect x="168" y="176" width="52" height="34" rx="6" fill="#4c8dd6"/><text x="194" y="198" fill="#fff">0.7</text>
</g>
<text x="194" y="282" text-anchor="middle" font-size="13" fill="currentColor">h = W<tspan font-size="9" dy="3">true</tspan><tspan dy="-3"> s</tspan></text>
<text x="194" y="299" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.62">dense activation</text>
<text x="194" y="312" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.62">(all the SAE sees)</text>
<!-- arrow 2 -->
<line x1="228" y1="174" x2="292" y2="174" stroke="currentColor" stroke-opacity="0.55" stroke-width="1.6" marker-end="url(#arrF)"/>
<text x="261" y="160" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.7">W<tspan font-size="8" dy="2">enc</tspan><tspan dy="-2">, ReLU</tspan></text>
<text x="261" y="192" text-anchor="middle" font-size="10.5" font-style="italic" fill="currentColor" opacity="0.7">sparse inference</text>
<!-- z column (4 cells) -->
<g font-size="13" font-weight="600" text-anchor="middle">
<rect x="302" y="100" width="44" height="34" rx="6" fill="#f08c00"/><text x="324" y="122" fill="#fff">1</text>
<rect x="302" y="138" width="44" height="34" rx="6" fill="none" stroke="currentColor" stroke-opacity="0.4"/><text x="324" y="160" fill="currentColor" opacity="0.5" font-weight="400">0</text>
<rect x="302" y="176" width="44" height="34" rx="6" fill="#f08c00"/><text x="324" y="198" fill="#fff">1</text>
<rect x="302" y="214" width="44" height="34" rx="6" fill="none" stroke="currentColor" stroke-opacity="0.4"/><text x="324" y="236" fill="currentColor" opacity="0.5" font-weight="400">0</text>
</g>
<text x="324" y="282" text-anchor="middle" font-size="13" fill="currentColor">z ≈ s</text>
<text x="324" y="299" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.62">SAE latents</text>
<text x="324" y="312" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.62">(sparse again!)</text>
<!-- arrow 3 -->
<line x1="354" y1="174" x2="418" y2="174" stroke="currentColor" stroke-opacity="0.55" stroke-width="1.6" marker-end="url(#arrF)"/>
<text x="387" y="160" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.7">W<tspan font-size="8" dy="2">dec</tspan></text>
<text x="387" y="192" text-anchor="middle" font-size="10.5" font-style="italic" fill="currentColor" opacity="0.7">reconstruct</text>
<!-- h-hat column (2 cells) -->
<g font-size="13" font-weight="600" text-anchor="middle">
<rect x="428" y="138" width="52" height="34" rx="6" fill="#4c8dd6" fill-opacity="0.42"/><text x="454" y="160" fill="currentColor">1.7</text>
<rect x="428" y="176" width="52" height="34" rx="6" fill="#4c8dd6" fill-opacity="0.42"/><text x="454" y="198" fill="currentColor">0.7</text>
</g>
<text x="454" y="282" text-anchor="middle" font-size="13" fill="currentColor">ĥ ≈ h</text>
<text x="454" y="299" text-anchor="middle" font-size="10.5" fill="currentColor" opacity="0.62">reconstruction</text>
<!-- brace hint: 4 -> 2 -> 4 -->
<text x="270" y="72" text-anchor="middle" font-size="11.5" fill="currentColor" opacity="0.62">4 features → 2 dims → 4 latents → 2 dims: the SAE is <tspan font-style="italic">wider</tspan> than its input</text>
</svg>
    </div>
  </div>
  <figcaption style="font-size:0.9em;opacity:0.75;margin-top:0.9rem;
                     max-width:70ch;margin-left:auto;margin-right:auto;text-align:left">
    <strong>Left:</strong> four feature directions d₁…d₄ share a two-dimensional residual
    stream, so they cannot all be orthogonal — that is superposition. On this input — the token <em>chat</em> in a French sentence — only
    d₁ (“cat”) and d₃ (“French”) are active (orange), and the network's activation is their sum
    <em>h</em> = (1.7, 0.7). Both <em>coordinates</em> of <em>h</em> are nonzero even though
    only two of the four <em>features</em> are on: coordinates are not features.
    <strong>Right:</strong> the same story as a pipeline. The network turns sparse
    <em>s</em> into dense <em>h</em>; the SAE, seeing only <em>h</em>, infers sparse latents
    <em>z</em> ≈ <em>s</em> and reconstructs ĥ ≈ <em>h</em>;
    latent 1 has become a “cat” detector, latent 3 a “French” detector. Its latent layer is wider than
    its input — the bottleneck is sparsity, not dimension.
  </figcaption>
</figure>


## Why the SAE latent space is *bigger*, not smaller

Notice the shape of the round trip. The network went $4 \to 2$ ($n$ features into $m < n$
dimensions), so the SAE must go $2 \to 4$: it needs **more latents than input dimensions**,
$M > m$ — one latent per feature it hopes to recover. A real SAE is the same picture scaled
up: in [Part 4](04_train_jumprelu_gemma3_1b.ipynb) we expand Gemma's $m = 1152$ residual dimensions into $M = 16384$ latents,
each of which we hope is as nameable as “cat” or “French” — in [Part 3](03_neuronpedia_dashboard.ipynb) you will meet one
that fires on fruit.

So an SAE is an **overcomplete** autoencoder. Its bottleneck is not "fewer numbers" — it has
*more* numbers than its input. Its bottleneck is that **most latents must be zero**: an
information bottleneck made of sparsity rather than dimensionality.

## What actually creates the sparsity?

Nothing in $z = \operatorname{ReLU}(W_{\text{enc}} h + b_{\text{enc}})$ forces $z$ to be
sparse. An overcomplete dictionary offers many ways to write $\hat h \approx h$, and most of
them smear the job across lots of small, dense activations. The ReLU only *permits* sparsity:
it lets a latent sit at exactly zero and keeps the rest non-negative.

What *rewards* sparsity is an explicit term in the loss:

$$
\mathcal{L} =
\underbrace{\lVert h - \hat h \rVert_2^2}_{\text{reconstruct } h}
\;+\;
\lambda\, \underbrace{\lVert z \rVert_1}_{\text{use few latents}}.
$$

The two terms fight. Reconstruction says "use enough latents to rebuild $h$"; the $L_1$
penalty makes every nonzero latent cost something, so reconstruction accuracy must *pay* for
each latent it turns on. (Parts 1–2 replace the $L_1$ term with better mechanisms — TopK,
JumpReLU — but the tug-of-war is the same.)

```{admonition} Aside — a preview of JumpReLU
:class: tip

There is a subtler problem that a plain ReLU cannot fix, visible right here in our example.
Take the most natural encoder — score each feature by dot product with its direction,
$a_i = d_i^\top h$ — and run it on $h = d_1 + d_3$:

$$
a = W_{\text{true}}^\top h = (1.70,\;\; 0.70,\;\; 1.68,\;\; 0.70).
$$

**Every score is positive**, so ReLU keeps all four: **dog** and **CAPS** appear to fire on a
French sentence about a cat. Because the four directions overlap, active features *leak* into
inactive detectors — and this time the leakage lands on the positive side, where clipping
negatives does nothing. (Contrast the previous section, where the antipodal $\pm 1$ embedding
conveniently made all interference negative.)

What separates truth from leakage here is not sign but **size**: active scores $\approx 1.7$,
leakage $= 0.7$. A ReLU with a per-latent *threshold* — exactly zero below $\theta$, **full
value** above it —

$$
\operatorname{JumpReLU}_\theta(a_i) = a_i \cdot \mathbb{1}[a_i > \theta]
$$

with any $\theta$ between $0.7$ and $1.68$ returns $(1.70,\; 0,\; 1.68,\; 0)$: the right two
features, at full strength. That is **JumpReLU**, the activation GemmaScope uses.
[Part 2](02_architectures_jumprelu.ipynb) shows why the threshold must be *learned*, and why
keeping the full value (rather than subtracting $\theta$) matters.
```

## The two networks, side by side again


This is the comparison the opening figure promised:

| | Bottleneck autoencoder (toy model) | Sparse autoencoder |
| --- | --- | --- |
| **Input** | sparse ground-truth features $s$ | dense network activation $h$ |
| **Middle layer** | *narrower* than input ($m < n$) | *wider* than input ($M > m$) |
| **What limits it** | dimension | sparsity penalty (most $z_i = 0$) |
| **Role of the ReLU** | clips negative interference in the output | permits exact zeros in the latents |
| **Shows** | how superposition arises | how to undo it |

$$
\boxed{\;\text{sparse features } s
\;\overset{\text{superposition (the network)}}{\longrightarrow}\;
\text{dense activation } h
\;\overset{\text{sparse inference (the SAE)}}{\longrightarrow}\;
\text{sparse latents } z \approx s\;}
$$

That reversal is the entire reason SAEs exist.

## Where we are

Three results to carry forward:

1. a **linear** bottleneck autoencoder can only project onto its top-$m$ principal subspace —
   it cannot store more features than dimensions;
2. a **ReLU** lets a bottleneck exploit *sparsity* to store more features than dimensions,
   paying only when features co-fire — that is superposition;
3. a **sparse autoencoder** runs that process in reverse: overcomplete, limited by a sparsity
   penalty rather than a dimension count, trained to un-mix a dense activation back into
   interpretable features.

**[Part 1](01_superposition_and_saes.ipynb)** makes all of this real: train the toy superposition model (watch five features
arrange themselves into a pentagon of shared directions), then train a ReLU SAE on it and
check that it recovers the ground-truth dictionary.
